In [1]:
#Amy Independent Research, Fall 2024
import networkx as nx
import osmnx as ox
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import ScrollZoomToggler
import matplotlib.pyplot as plt
import shapely
from glob import glob
from datetime import timedelta
import osmnx.settings as settings
import osmnx.features as features

ox.__version__

settings.cache_folder = "/tmp/cache"

In [2]:
file_path = "/Users/amyxqc/Desktop/amy_bikenycfall2024/Data/CitiBike/202101-citibike-tripdata_1.csv"

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

/var/folders/30/jfztb8191jzff8_xz9n0d59m0000gn/T/ipykernel_40270/4193616194.py:4: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [26]:
# Parse date columns with single-digit handling
df['started_at'] = pd.to_datetime(df['started_at'], errors='coerce', dayfirst=False)
df['ended_at'] = pd.to_datetime(df['ended_at'], errors='coerce', dayfirst=False)

# Drop rows with invalid dates
df = df.dropna(subset=['started_at', 'ended_at'])

# Ensure all dates are in 2021
df = df[df['started_at'].dt.year == 2021]

# Extract year, month, week number, and day of week
df['year'] = df['started_at'].dt.year
df['month'] = df['started_at'].dt.month
df['week_number'] = df['started_at'].dt.isocalendar().week
df['day_of_week'] = df['started_at'].dt.weekday  # 0=Monday, 6=Sunday


In [27]:
# Define bounding boxes for each borough
borough_bounds = {
    'Manhattan': {'lat_min': 40.70, 'lat_max': 40.88, 'lng_min': -74.02, 'lng_max': -73.90},
    'Brooklyn': {'lat_min': 40.57, 'lat_max': 40.73, 'lng_min': -74.04, 'lng_max': -73.85},
    'Queens': {'lat_min': 40.54, 'lat_max': 40.80, 'lng_min': -73.95, 'lng_max': -73.70},
    'Bronx': {'lat_min': 40.79, 'lat_max': 40.91, 'lng_min': -73.93, 'lng_max': -73.80},
    'Staten Island': {'lat_min': 40.49, 'lat_max': 40.65, 'lng_min': -74.25, 'lng_max': -74.05}
}

def get_borough(lat, lng):
    for borough, bounds in borough_bounds.items():
        if bounds['lat_min'] <= lat <= bounds['lat_max'] and bounds['lng_min'] <= lng <= bounds['lng_max']:
            return borough
    return 'Other'

# Apply the get_borough function to determine the borough based on latitude and longitude
df['borough'] = df.apply(lambda row: get_borough(row['start_lat'], row['start_lng']), axis=1)

In [28]:
# Calculate trip duration in seconds and convert to minutes
df['trip_duration'] = (pd.to_datetime(df['ended_at']) - pd.to_datetime(df['started_at'])).dt.total_seconds()
df['trip_duration_min'] = df['trip_duration'] / 60

# Average trip duration for all days
avg_trip_duration_all = df['trip_duration_min'].mean()

# Average trip duration for weekdays (Monday to Friday)
avg_trip_duration_weekday = df[df['started_at'].dt.weekday < 5]['trip_duration_min'].mean()

# Average trip duration for weekends (Saturday and Sunday)
avg_trip_duration_weekend = df[df['started_at'].dt.weekday >= 5]['trip_duration_min'].mean()

In [29]:
# Aggregate weekly data
weekly_aggregated = df.groupby(['year', 'month', 'week_number', 'borough']).agg(
    total_trips=('ride_id', 'count'),
    weekday_trips=('day_of_week', lambda x: (x < 5).sum()),  # Weekdays (Mon-Fri)
    weekend_trips=('day_of_week', lambda x: (x >= 5).sum()),  # Weekends (Sat-Sun)
    electric_bike_rides=('rideable_type', lambda x: (x == 'electric_bike').sum()),
    classic_bike_rides=('rideable_type', lambda x: (x == 'classic_bike').sum()),
    member_rides=('member_casual', lambda x: (x == 'member').sum()),
    casual_rides=('member_casual', lambda x: (x == 'casual').sum()),
    avg_trip_duration=('trip_duration_min', 'mean'),  # Average trip duration
    unique_start_stations=('start_station_id', 'nunique'),
    unique_end_stations=('end_station_id', 'nunique')
).reset_index()

# Calculate proportions and averages
weekly_aggregated['electric_bike_proportion'] = weekly_aggregated['electric_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['classic_bike_proportion'] = weekly_aggregated['classic_bike_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['member_proportion'] = weekly_aggregated['member_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['casual_proportion'] = weekly_aggregated['casual_rides'] / weekly_aggregated['total_trips']
weekly_aggregated['avg_daily_trips'] = weekly_aggregated['total_trips'] / 7
weekly_aggregated['avg_weekday_trips'] = weekly_aggregated['weekday_trips'] / 5
weekly_aggregated['avg_weekend_trips'] = weekly_aggregated['weekend_trips'] / 2

# Ensure data exists
if weekly_aggregated.empty:
    print("Error: `weekly_aggregated` is empty. Check your data and aggregation logic.")
    exit()

# Weighted average calculation function
def weighted_average(data, value_column, weight_column):
    """Calculate weighted average."""
    weights = data[weight_column]
    values = data[value_column]
    if weights.sum() == 0:
        return 0
    return np.average(values, weights=weights)

Error: `weekly_aggregated` is empty. Check your data and aggregation logic.


: 

In [23]:
# Calculate NYC Total
nyc_total = weekly_aggregated.groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum')
).reset_index()

nyc_total['avg_trip_duration'] = nyc_total.apply(
    lambda row: weighted_average(weekly_aggregated, 'avg_trip_duration', 'total_trips'),
    axis=1
)
nyc_total['borough'] = 'NYC Total'

In [24]:
# Calculate Non-Manhattan Total
non_manhattan_total = weekly_aggregated[weekly_aggregated['borough'] != 'Manhattan'].groupby(['year', 'month', 'week_number']).agg(
    total_trips=('total_trips', 'sum'),
    weekday_trips=('weekday_trips', 'sum'),
    weekend_trips=('weekend_trips', 'sum'),
    electric_bike_rides=('electric_bike_rides', 'sum'),
    classic_bike_rides=('classic_bike_rides', 'sum'),
    member_rides=('member_rides', 'sum'),
    casual_rides=('casual_rides', 'sum'),
    unique_start_stations=('unique_start_stations', 'sum'),
    unique_end_stations=('unique_end_stations', 'sum')
).reset_index()

non_manhattan_total['avg_trip_duration'] = non_manhattan_total.apply(
    lambda row: weighted_average(weekly_aggregated[weekly_aggregated['borough'] != 'Manhattan'], 'avg_trip_duration', 'total_trips'),
    axis=1
)
non_manhattan_total['borough'] = 'Non-Manhattan Total'

In [25]:
# Combine the original data with NYC and Non-Manhattan totals
updated_weekly_aggregated = pd.concat([weekly_aggregated, nyc_total, non_manhattan_total], ignore_index=True)

# Sort the DataFrame
updated_weekly_aggregated = updated_weekly_aggregated.sort_values(by=['year', 'month', 'week_number', 'borough']).reset_index(drop=True)

# Display updated DataFrame
print(updated_weekly_aggregated)

    year  month  week_number              borough  total_trips  weekday_trips  \
0   2021      1            1                Bronx          413            324   
1   2021      1            1             Brooklyn        29635          20775   
2   2021      1            1            Manhattan       205760         150601   
3   2021      1            1            NYC Total       235808         171700   
4   2021      1            1  Non-Manhattan Total        30048          21099   
5   2021      1            2                Bronx          484            346   
6   2021      1            2             Brooklyn        33540          23335   
7   2021      1            2            Manhattan       229788         166959   
8   2021      1            2            NYC Total       263812         190640   
9   2021      1            2  Non-Manhattan Total        34024          23681   
10  2021      1            3                Bronx          321            246   
11  2021      1            3